# ICT Backtest + P39 Volume Analysis — Google Colab runner

Runs the **full 2022–2025 backtest** and the **P39 tick-volume analysis** directly on the data
already in your Google Drive. **Nothing to upload.**

### Before you run (one-time, ~2 min): get a GitHub token
The code lives in a *private* repo, so Colab needs a read-only token to fetch it.
1. On your phone open **github.com** → tap your avatar → **Settings**
2. Scroll down → **Developer settings** → **Personal access tokens** → **Fine-grained tokens** → **Generate new token**
3. Name it anything, Expiration 7 days. Under **Repository access** pick **Only select repositories → ThabisoCollinSengane/Ict**.
4. Under **Permissions → Repository permissions → Contents**, set **Read-only**. Generate, then **copy** the token (starts `github_pat_…`).

Then: **Runtime ▸ Run all** (top menu) and paste the token when the 3rd cell asks. That's it.

_Every table in the P39 report is split IS (2022) vs OOS (2024). Results are also saved to a new_
_`ICT_results` folder in your Drive so Claude can read them back._


### 1. Mount your Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


### 2. Fetch the code (paste your GitHub token when prompted — input is hidden)


In [ ]:
import os, glob, getpass, subprocess
REPO   = 'ThabisoCollinSengane/Ict'
BRANCH = 'p39-volume-analysis'
TOKEN  = getpass.getpass('Paste GitHub token, then Enter: ').strip()
if os.path.isdir('/content/Ict'):
    subprocess.run(['rm','-rf','/content/Ict'])
url = f'https://{TOKEN}@github.com/{REPO}.git'
r = subprocess.run(['git','clone','--branch',BRANCH,'--depth','1',url,'/content/Ict'],
                   capture_output=True, text=True)
# never echo the token back
print(r.stderr[-800:].replace(TOKEN,'***') if r.stderr else 'cloned')
assert os.path.isdir('/content/Ict/scripts'), 'Clone failed — re-check the token + repo access.'
os.chdir('/content/Ict')
print('Repo ready at', os.getcwd())


### 3. Install the Python dependencies (~1 min)


In [ ]:
import subprocess
p = subprocess.run(['pip','install','-q','-r','requirements.txt'], capture_output=True, text=True)
print(p.stdout[-400:]); print(p.stderr[-400:])
print('dependencies installed')


### 4. Locate your data folders in Drive
Auto-finds the M1 folder and the (separate) tick folder by name. If either shows 0 zips,
edit the `M1_DIR` / `TICK_DIR` line to the exact path and re-run this cell.


In [ ]:
import glob, os
def find_dir(tokens, must_glob):
    # direct children of My Drive first (fast), then a recursive fallback
    for d in glob.glob('/content/drive/MyDrive/*/'):
        b = os.path.basename(d.rstrip('/')).lower()
        if all(t in b for t in tokens) and glob.glob(os.path.join(d, must_glob)):
            return d.rstrip('/')
    for root, _dirs, _files in os.walk('/content/drive/MyDrive'):
        b = os.path.basename(root).lower()
        if all(t in b for t in tokens) and glob.glob(os.path.join(root, must_glob)):
            return root
    return None

M1_DIR   = find_dir(['backtesting','data'], 'HISTDATA_*_M1*.zip')
TICK_DIR = find_dir(['tick','data'],        'HISTDATA_*_T*.zip')

print('M1 folder  :', M1_DIR)
print('  M1 zips  :', len(glob.glob(f'{M1_DIR}/HISTDATA_*_M1*.zip')) if M1_DIR else 0)
print('Tick folder:', TICK_DIR)
print('  tick zips:', len(glob.glob(f'{TICK_DIR}/HISTDATA_*_T*.zip')) if TICK_DIR else 0)
assert M1_DIR and TICK_DIR, 'One folder was not found — set the path manually above and re-run.'


### 5. Prepare the M1 data
Unzips + converts (incl. UDXUSD MT→ASCII) + renames every M1 file into `data/histdata/`.
Look for `OK — runnable` on the year lines.


In [ ]:
!python scripts/prepare_histdata.py "{M1_DIR}"


### 6. Run the full backtest (current algorithm, 2022–2025)
This is the R429M / PF 4.47 / MaxDD −12.95% run — and it produces the trade dump P39 needs.
The summary is at the very end of the output.


In [ ]:
import subprocess
res = subprocess.run(['python','run_backtest_histdata.py','--years','2022','2023','2024','2025'],
                     capture_output=True, text=True)
open('/content/backtest_output.txt','w').write(res.stdout)
print(res.stdout[-9000:])
if res.returncode != 0:
    print('--- STDERR ---'); print(res.stderr[-3000:])


### 7. P39 — aggregate the tick data (the slow step; several minutes)
Streams each tick month into compact M5 tick/delta counts. 0-byte / `.txt` months are flagged, not fatal.


In [ ]:
!python scripts/p39_volume_analysis.py aggregate "{TICK_DIR}"


### 8. P39 — analyse and write the report


In [ ]:
!python scripts/p39_volume_analysis.py analyse


### 9. Save results to your Drive
Copies the backtest summary, the trade dump, and `p39_volume_report.md` to **Drive/ICT_results**.


In [ ]:
import shutil, os
OUT = '/content/drive/MyDrive/ICT_results'
os.makedirs(OUT, exist_ok=True)
for f in ['data/p39_volume_report.md',
          'data/histdata/trades_dump.csv',
          '/content/backtest_output.txt']:
    if os.path.exists(f):
        shutil.copy(f, OUT); print('saved ->', os.path.basename(f))
    else:
        print('missing (did an earlier cell error?):', f)
print()
print('DONE. Results are in your Drive: ICT_results/')
print('Tell Claude \'results are ready\' and it will read them from your Drive.')


---
**When it finishes:** come back to the Claude chat and say *"results are ready"*. Claude will read
`ICT_results/p39_volume_report.md` and the backtest summary straight from your Drive and go through
the verdict with you. You can also open the report yourself in Drive.
